In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.isaaa.org/animalbiotechdatabase/ngtlist/ngt/default.asp?NGTID=4&RegProcess=1"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table')
rows = table.find_all('tr')

data = []

for row in rows:
    cells = row.find_all('td')
    
    # Skip rows that don’t match expected structure
    if len(cells) < 5:
        continue

    # Clean extraction with safeguards
    year = cells[0].get_text(" ", strip=True)
    country = cells[1].get_text(" ", strip=True)

    # Animal parsing (robust)
    animal_common = ""
    animal_scientific = ""

    if cells[2]:
        # Get all text
        full_text = cells[2].get_text(" ", strip=True)

        # Scientific name if exists
        em = cells[2].find('em')
        if em:
            animal_scientific = em.get_text(strip=True)
            animal_common = full_text.replace(animal_scientific, "").strip(" ()")
        else:
            animal_common = full_text

    gene = cells[3].get_text(" ", strip=True)

    # Details URL
    link = cells[4].find('a')
    details_url = ""
    if link and link.get('href'):
        details_url = "https://www.isaaa.org" + link['href']

    data.append({
        "year": year,
        "country": country,
        "animal_common": animal_common,
        "animal_scientific": animal_scientific,
        "gene": gene,
        "details_url": details_url
    })

df4 = pd.DataFrame(data)
print("Rows scraped:", len(df4))
df4.head()

Rows scraped: 3


,year,country,animal_common,animal_scientific,gene,details_url
0,Year,Country of Determination,Species,,Genes,
1,2021,Argentina,Cattle\r\n,Bos taurus x Bos taurus indicus,MSTN,https://www.isaaa.org/animalbiotechdatabase/de...
2,,Brazil,Cattle\r\n,Bos taurus x Bos taurus indicus,MSTN,https://www.isaaa.org/animalbiotechdatabase/de...


In [2]:
df4['Institute'] = ""
df4['Editing_Method'] = ""
df4['Delivery'] = ""
df4['Foreign_DNA'] = ""
df4['Regulatory_Process'] = ""

In [3]:
import time

def get_field_value(soup, field_name):
    tds = soup.find_all('td')
    
    for i, td in enumerate(tds):
        text = td.get_text(" ", strip=True).lower()
        
        if field_name.lower() in text:
            if i + 1 < len(tds):
                return tds[i+1].get_text(" ", strip=True)
    
    return ""

for idx in df4.index:
    url = df4.loc[idx, 'details_url']
    
    if not url:
        continue

    print(f"({idx+1}/{len(df4)}) Fetching {url}")
    
    resp = requests.get(url)
    detail_soup = BeautifulSoup(resp.text, 'html.parser')

    df4.loc[idx, 'Institute'] = get_field_value(detail_soup, 'institute')
    df4.loc[idx, 'Editing_Method'] = get_field_value(detail_soup, 'editing method')
    df4.loc[idx, 'Delivery'] = get_field_value(detail_soup, 'delivery')
    df4.loc[idx, 'Foreign_DNA'] = get_field_value(detail_soup, 'foreign dna')
    df4.loc[idx, 'Regulatory_Process'] = get_field_value(detail_soup, 'undergone regulatory')

    time.sleep(0.4)

(2/3) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=15&RegProcess=1
(3/3) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=14&RegProcess=1


In [4]:
df4.to_csv("new_isaa_clean.csv", index=False)
print("Saved clean dataset:", len(df4), "rows")

Saved clean dataset: 3 rows


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.isaaa.org/animalbiotechdatabase/ngtlist/ngt/default.asp?NGTID=4&RegProcess=0"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table')
rows = table.find_all('tr')

data = []

for row in rows:
    cells = row.find_all('td')
    
    if len(cells) < 5:
        continue

    year = cells[0].get_text(" ", strip=True)
    country = cells[1].get_text(" ", strip=True)

    # Robust animal parsing
    animal_common = ""
    animal_scientific = ""

    full_text = cells[2].get_text(" ", strip=True)
    em = cells[2].find('em')

    if em:
        animal_scientific = em.get_text(strip=True)
        animal_common = full_text.replace(animal_scientific, "").strip(" ()")
    else:
        animal_common = full_text

    gene = cells[3].get_text(" ", strip=True)

    link = cells[4].find('a')
    details_url = ""
    if link and link.get('href'):
        details_url = "https://www.isaaa.org" + link['href']

    data.append({
        "year": year,
        "country": country,
        "animal_common": animal_common,
        "animal_scientific": animal_scientific,
        "gene": gene,
        "details_url": details_url
    })

df5 = pd.DataFrame(data)
print("Rows scraped:", len(df5))
df5.head()

Rows scraped: 27


,year,country,animal_common,animal_scientific,gene,details_url
0,Year,Country of 1st Author,Species,,Genes,
1,2023,Japan,Chicken\r\n,Gallus gallus,OVM,https://www.isaaa.org/animalbiotechdatabase/de...
2,,United States,Channel catfish\r\n,Ictalurus punctatus,fsh,https://www.isaaa.org/animalbiotechdatabase/de...
3,2021,China,Cattle\r\n,Bos taurus,"SRY, ZFY, DDX3Y, Eif2s3y",https://www.isaaa.org/animalbiotechdatabase/de...
4,,New Zealand,Cattle\r\n,Bos taurus,PMEL,https://www.isaaa.org/animalbiotechdatabase/de...


In [6]:
df5['Institute'] = ""
df5['Editing_Method'] = ""
df5['Delivery'] = ""
df5['Foreign_DNA'] = ""
df5['Regulatory_Process'] = ""

In [7]:
import time

def get_field_value(soup, field_name):
    tds = soup.find_all('td')
    
    for i, td in enumerate(tds):
        text = td.get_text(" ", strip=True).lower()
        
        if field_name.lower() in text:
            if i + 1 < len(tds):
                return tds[i+1].get_text(" ", strip=True)
    
    return ""

for idx in df5.index:
    url = df5.loc[idx, 'details_url']
    
    if not url:
        continue

    print(f"({idx+1}/{len(df5)}) Fetching {url}")
    
    resp = requests.get(url)
    detail_soup = BeautifulSoup(resp.text, 'html.parser')

    df5.loc[idx, 'Institute'] = get_field_value(detail_soup, 'institute')
    df5.loc[idx, 'Editing_Method'] = get_field_value(detail_soup, 'editing method')
    df5.loc[idx, 'Delivery'] = get_field_value(detail_soup, 'delivery')
    df5.loc[idx, 'Foreign_DNA'] = get_field_value(detail_soup, 'foreign dna')
    df5.loc[idx, 'Regulatory_Process'] = get_field_value(detail_soup, 'undergone regulatory')

    time.sleep(0.4)

(2/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=86&RegProcess=0
(3/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=92&RegProcess=0
(4/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=53&RegProcess=0
(5/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=49&RegProcess=0
(6/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=168&RegProcess=0
(7/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=2&RegProcess=0
(8/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=54&RegProcess=0
(9/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=33&RegProcess=0
(10/27) Fetching https://www.isaaa.org/animalbiotechdatabase/details/default.asp?ABDEntryID=26&RegProcess=0
(11/27) Fetching https://www.isaaa.o

In [8]:
df5.to_csv("./new_isaa_ngt4s0_clean.csv", index=False)
print("Saved:", len(df5), "rows")

Saved: 27 rows
